In [ ]:
# ============================================================
# CONFIG — edit only this cell
# ============================================================
# 前提条件: docs/POSEBUSTERS_PREREQUISITES.md を必ず読んでください
#
# インストール:
#   pip install "mdatools[posebusters] @ git+https://github.com/rkakamilan/md-analysis-tools-public.git"
#
from pathlib import Path
from mdatools.config import AnalysisConfig

cfg = AnalysisConfig(
    ligand_resname = "UNK",   # トポロジー内のリガンド残基名
    dt_ns          = 2.0,
    output_dir     = Path("./hbond_results"),
    figures_dir    = Path("./figures"),
)

# ----------------------------------------------------------
# 【必須】リガンドの正しい結合次数を持つ SMILES または SDF
#   PDB 形式は結合次数を持たないため、これがないと
#   PoseBusters の化学妥当性チェックが失敗します。
#   詳細: docs/POSEBUSTERS_PREREQUISITES.md
# ----------------------------------------------------------
LIGAND_SMILES = ""   # 例: "CC1=CC=CC=C1"  ← あなたのリガンドの SMILES
LIGAND_SDF    = None # SMILES の代わりに SDF を使う場合: Path("ligand.sdf")

# 【必須】溶媒・イオンを除去した タンパク質単体の PDB
PROTEIN_PDB = Path("protein_clean.pdb")

# 軌跡ファイル（MDAnalysis backend の場合）
TOPOLOGY   = Path("../run01/equilibrating_topology.pdb")
TRAJECTORY = Path("../run01/trajectory.xtc")

# または PSE ファイル（PyMOL backend の場合）
PSE_PATH   = None  # Path("../run01/traj.pse")

SAMPLE_NAME = "run01"
PB_FRAMES_DIR = Path("./pb_frames") / SAMPLE_NAME

# PoseBusters オプション
FULL_REPORT = False   # True にすると 35 チェック（低速）
FRAME_STEP  = 1       # フレームのサンプリング間隔
MAX_FRAMES  = None    # None = 全フレーム
# ============================================================

## Step 1: フレームエクスポート

軌跡の各フレームを個別の PDB ファイルとして出力します。
溶媒・イオンは自動的に除去されます。

In [ ]:
from mdatools.posebusters import FrameExporter

cfg.make_dirs()
exporter = FrameExporter(cfg)

if PSE_PATH is not None and PSE_PATH.exists():
    # PyMOL backend
    pdb_files = exporter.export_from_pse(
        pse_path=PSE_PATH,
        output_dir=PB_FRAMES_DIR,
        prefix=f"snapshot_{SAMPLE_NAME}",
        max_frames=MAX_FRAMES or 100,
        step=FRAME_STEP,
    )
else:
    # MDAnalysis backend (preferred)
    from mdatools.universe import load_and_align
    u = load_and_align(TOPOLOGY, TRAJECTORY, cfg)
    pdb_files = exporter.export_from_universe(
        u,
        output_dir=PB_FRAMES_DIR,
        prefix=f"snapshot_{SAMPLE_NAME}",
        step=FRAME_STEP,
        max_frames=MAX_FRAMES,
    )

print(f"Exported {len(pdb_files)} frames → {PB_FRAMES_DIR}")

## Step 2: PoseBusters バッチ実行

⚠️ **LIGAND_SMILES または LIGAND_SDF が未設定の場合はここでエラーになります。**

設定方法は `docs/POSEBUSTERS_PREREQUISITES.md` を参照してください。

In [ ]:
from mdatools.posebusters import PoseBustersValidator

# SMILES が未設定の場合はここで明示的なエラーを出す
if not LIGAND_SMILES and LIGAND_SDF is None:
    raise ValueError(
        "LIGAND_SMILES または LIGAND_SDF を設定してください。\n"
        "詳細: docs/POSEBUSTERS_PREREQUISITES.md"
    )

validator = PoseBustersValidator(
    cfg,
    ligand_smiles=LIGAND_SMILES or None,
    ligand_sdf=LIGAND_SDF,
    full_report=FULL_REPORT,
)

pb_result = validator.run(
    pdb_dir=PB_FRAMES_DIR,
    protein_path=PROTEIN_PDB,
    prefix=f"snapshot_{SAMPLE_NAME}",
)

print(f"Pass rate: {pb_result.pass_rate:.1%}  ({pb_result.results['all_pass'].sum()}/{len(pb_result.results)} frames)")

# 結果を保存
cfg.output_dir.mkdir(exist_ok=True)
pb_result.results.to_csv(cfg.output_dir / f"posebusters_{SAMPLE_NAME}.csv", index=False)

## Step 3: 上位フレーム（best poses）

In [ ]:
top = pb_result.top_frames(10)
print("Top 10 frames by energy_ratio:")
display(top)

print("\nFailed check summary:")
display(pb_result.failed_checks_summary())

## Step 4: 可視化

In [ ]:
import matplotlib.pyplot as plt
from mdatools.plotting.posebusters_plots import (
    plot_pb_pass_rate,
    plot_pb_check_heatmap,
    plot_pb_score_timeseries,
)

# チェック別合格率
plot_pb_pass_rate(pb_result, save_path=cfg.figures_dir / f"pb_pass_rate_{SAMPLE_NAME}.png")
plt.show()

# フレーム × チェック ヒートマップ
plot_pb_check_heatmap(pb_result, save_path=cfg.figures_dir / f"pb_heatmap_{SAMPLE_NAME}.png")
plt.show()

# タイムシリーズ（H-bond と重ねる場合）
import pandas as pd
hbond_csv = cfg.output_dir / f"hbond_all_events_{SAMPLE_NAME}.csv"
hbond_events = pd.read_csv(hbond_csv) if hbond_csv.exists() else None

plot_pb_score_timeseries(
    pb_result,
    hbond_events=hbond_events,
    dt_ns=cfg.dt_ns,
    save_path=cfg.figures_dir / f"pb_timeseries_{SAMPLE_NAME}.png",
)
plt.show()